# Supervised analysis (_Glass_)

In [14]:
from experiments.utils.constants import RANDOM_SEED, SOM_LEARNING_RATE_DECAY_FN

VERBOSE = True
DATASET_NAME = "Glass"
DATASET_ID = 42

EXPORT_MODE = False
EXPORT_DIR = "_exports"

print(DATASET_NAME)

Glass


## Dataset

In [15]:
# fetch dataset
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=DATASET_ID)
X = dataset.data.features.values
y = dataset.data.targets.values.ravel()
print(f"Dataset shape: {X.shape}, {y.shape}")

Dataset shape: (214, 9), (214,)


In [16]:
# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
# encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)
label_decoder = {i: label for i, label in enumerate(le.classes_)}
num_unique_y = len(le.classes_)
print(f"Classes: {num_unique_y}")

Classes: 6


## Re-create model

In [18]:
from minisom_representation import calc_som_hyparams, SomRepresentation

In [19]:
# hyperparameters
recommended_params = calc_som_hyparams(X_scaled, initial_sigma_factor=3.0)
print("Recommended SOM parameters:", recommended_params)
d1, d2, sigma = map(recommended_params.get, ("d1", "d2", "sigma"))
decay_function = SOM_LEARNING_RATE_DECAY_FN
epoch = 27

Recommended SOM parameters: {'d1': 9, 'd2': 10, 'sigma': 3.33}


In [20]:
# fit SOM representation
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE, decay_function=decay_function) \
    .fit_online(X_scaled, num_iteration=epoch)

 [ 5778 / 5778 ] 100% - 0:00:00 left 
 quantization error: 1.1077913729377833

 An SOM representation has been fitted as follows:
------------------------------------------------------- 

Fit strategy: online 

Hyperparameters of SOM: 

{'input_len': 9, 'x': 9, 'y': 10, 'sigma': 3.33, 'topology': 'rectangular', 'learning_rate': 0.5, 'decay_function': 'linear_decay_to_zero', 'sigma_decay_function': 'asymptotic_decay', 'neighborhood_function': 'gaussian', 'activation_distance': 'euclidean', 'random_seed': 42, 'num_iteration': 27, 'use_epochs': True, 'random_order': True, 'verbose': True} 

Quality of SOM: 

Quantization Error (QE):	1.1077913729377833
Topographic Error (TE): 	0.004672897196261682


## Inspection

In [21]:
from utils.plotting import PlotlyHelperArgs

In [22]:
# create Basin
from lilypond import Basin
basin = Basin.from_som_representation(som_rep, random_seed=RANDOM_SEED, verbose=VERBOSE)

In [23]:
# lilypond visual
basin.pond() \
    .rhizome_layer() \
    .pad_layer() \
    .petal_layer() \
    .visualize(width=800, height=400);

## Extra figures

In [24]:
plot_args = dict(
    **PlotlyHelperArgs.Figsize(w=800, h=750),
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    showlegend=False
)

In [25]:
import plotly.express as px
palette = px.colors.qualitative.Light24
marker_colors = [palette[val % len(palette)] for val in y_encoded]

In [26]:
fig1 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer() \
    .attraction_layer(X_scaled, marker=dict(color=marker_colors, symbol="star-diamond", size=20, opacity=.8), name="Projection of training data colored by class") \
    .visualize(**plot_args);

In [27]:
if EXPORT_MODE:
    fig1.write_image(EXPORT_DIR + "/03_02_lilypond_01.png")

---

### The below cells are not part of the experiment. They are used to persist the data and register the model in Databricks and Bianor for further interactive investigation.

---

## Preparation

In [28]:
MODEL_REGISTRATION_MODE = False
DATASET_PERSIST_MODE = True

In [29]:
import pandas as pd
import mlflow

from dotenv import load_dotenv
from utils.databricks_util import get_spark, get_catalog_path, CATALOG, SCHEMA

In [30]:
load_dotenv()
spark = get_spark()

## Persist data in Databricks

In [31]:
TABLE_NAME = f"T_{DATASET_NAME}".lower()
TABLE_PATH = get_catalog_path(TABLE_NAME)
print(TABLE_PATH)

workspace.lilypond_experiments.t_glass


In [32]:
if DATASET_PERSIST_MODE:
	# persist full dataset as a managed table
    spark.createDataFrame(
		pd.DataFrame(X) \
			.assign(label=y_encoded) \
			.reset_index(names="id") \
        	.rename(columns={"label": "class"})
	).write \
		.mode("overwrite") \
		.saveAsTable(TABLE_PATH)

In [33]:
data_dbdf = spark.read \
    .table(TABLE_PATH)
data_dbdf.show(5)

+---+-------+-----+----+----+-----+----+----+---+---+-----+
| id|      0|    1|   2|   3|    4|   5|   6|  7|  8|class|
+---+-------+-----+----+----+-----+----+----+---+---+-----+
|  0|1.52101|13.64|4.49| 1.1|71.78|0.06|8.75|0.0|0.0|    0|
|  1|1.51761|13.89| 3.6|1.36|72.73|0.48|7.83|0.0|0.0|    0|
|  2|1.51618|13.53|3.55|1.54|72.99|0.39|7.78|0.0|0.0|    0|
|  3|1.51766|13.21|3.69|1.29|72.61|0.57|8.22|0.0|0.0|    0|
|  4|1.51742|13.27|3.62|1.24|73.08|0.55|8.07|0.0|0.0|    0|
+---+-------+-----+----+----+-----+----+----+---+---+-----+
only showing top 5 rows


In [34]:
# separate variables
primary_key = ['id']
target = ['class']
data_df = data_dbdf.toPandas()
features = data_df.columns.difference((primary_key + target), sort=False).tolist()
assert primary_key[0] not in features, "Primary key shall not be in Features"
assert all(t not in features for t in target), "Targets must not be in Features"
print("Primary key:", primary_key)
print("Targets:", target)
print("Features:", features)

Primary key: ['id']
Targets: ['class']
Features: ['0', '1', '2', '3', '4', '5', '6', '7', '8']


In [35]:
# create feature dataframe
feature_dbdf = data_dbdf.select(primary_key + features)
feature_dbdf.show(5)

+---+-------+-----+----+----+-----+----+----+---+---+
| id|      0|    1|   2|   3|    4|   5|   6|  7|  8|
+---+-------+-----+----+----+-----+----+----+---+---+
|  0|1.52101|13.64|4.49| 1.1|71.78|0.06|8.75|0.0|0.0|
|  1|1.51761|13.89| 3.6|1.36|72.73|0.48|7.83|0.0|0.0|
|  2|1.51618|13.53|3.55|1.54|72.99|0.39|7.78|0.0|0.0|
|  3|1.51766|13.21|3.69|1.29|72.61|0.57|8.22|0.0|0.0|
|  4|1.51742|13.27|3.62|1.24|73.08|0.55|8.07|0.0|0.0|
+---+-------+-----+----+----+-----+----+----+---+---+
only showing top 5 rows


In [36]:
# create feature table
from databricks.feature_engineering import FeatureEngineeringClient
FEATURE_TABLE_NAME = f"{TABLE_NAME}_feature"
FEATURE_TABLE_PATH = get_catalog_path(FEATURE_TABLE_NAME)
print(FEATURE_TABLE_PATH)

workspace.lilypond_experiments.t_glass_feature


In [37]:
if DATASET_PERSIST_MODE:
	feClient = FeatureEngineeringClient()
	feClient.create_table(
		name=FEATURE_TABLE_PATH,
		primary_keys=primary_key,
		df=feature_dbdf,
		description=f"{DATASET_NAME} features (original)",
		tags={"source": "bronze", "format": "delta"}
	)

2026/09/16 00:47:17 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['id'] of table 'workspace.lilypond_experiments.t_glass_feature' to NOT NULL.
2026/09/16 00:47:20 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['id'] on table 'workspace.lilypond_experiments.t_glass_feature'.
2026/09/16 00:47:36 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'workspace.lilypond_experiments.t_glass_feature'.


## Register representation model in Databricks

In [ ]:
MODEL_NAME = f"som-{DATASET_NAME.lower()}"
MODEL_PATH = get_catalog_path(MODEL_NAME)
EXPERIMENT_NAME = f"/Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_{DATASET_NAME}"
print(MODEL_PATH, EXPERIMENT_NAME)

In [ ]:
if MODEL_REGISTRATION_MODE:

	mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

	class MLflowSomModelWrapper(mlflow.pyfunc.PythonModel):
		from typing import Any

		def __init__(self, model:SomRepresentation, scaler):
			self.model = model
			self.scaler = scaler
		def predict(self, context, model_input, params: dict[str, Any] | None = None):
			"""First transforms the input data via scaler, then predicts the winner node of the SOM."""
			return [self.model.som.winner(x) for x in self.scaler.transform(model_input.to_numpy())]

	with mlflow.start_run():
		model = MLflowSomModelWrapper(som_rep, scaler)

		mlflow.log_metric("QE", som_rep.quantization_error)
		mlflow.log_metric("TE", som_rep.topographic_error)

		mlflow.pyfunc.log_model(
			python_model=model,
			name=MODEL_NAME,
			input_example=pd.DataFrame(X_scaled[:3]),
			pip_requirements=[
				"numpy",
				"pandas",
				"scikit-learn==1.5.2",
				"mlflow",
				"minisom",
			],
			registered_model_name=MODEL_PATH
		)

else: print("Skipping MLflow model registration.")

In [ ]:
version = 1

registered_model = f"{MODEL_NAME}/{version}"
print(registered_model)

registered_model_location = get_catalog_path(registered_model)
print(registered_model_location)

## Register metadata in Bianor

In [40]:
from bianor_databricks_kit import BianorRecordManager
bianor_recorder = BianorRecordManager(catalog=CATALOG, schema=SCHEMA, spark=spark)

In [ ]:
if MODEL_REGISTRATION_MODE:

		# first registration
		# bianor_recorder.new_representation(name=f"{DATASET_NAME} Representation", som_model_location=registered_model_location, features_location="c.s.t")

		rep_id = "7fa8afac-8267-47b0-8dd4-458169fe4994"

		# update existing
		bianor_recorder.update_representation(
			representation_id=rep_id,
			features_location=FEATURE_TABLE_PATH
		)

		# register projection layers

		class_names = [f"type_{c}" for c in le.classes_]
		sample_locations = [f"V_{DATASET_NAME}_class_{cn}".lower() for cn in class_names]
		colors = palette[:len(class_names)]
		names = [f"Training data - {cn}" for cn in class_names]

		for color, name, sample_loc in zip(colors, names, sample_locations):
			marker_dict = dict(
				color=color,
				symbol="star-diamond",
				opacity=0.9
			)

			proj_id = bianor_recorder.new_projection_layer(
				name=name,
				marker_dict=marker_dict,
				samples_location=get_catalog_path(sample_loc)
			)

			bianor_recorder.new_map_representation_projection_layer(representation_id=rep_id, projection_layer_id=proj_id)


else: print("Skipping Bianor metadata registration.")